In [4]:
from dotenv import load_dotenv
load_dotenv()
import os
import json

In [2]:
from openai import OpenAI



openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1"#"https://api.groq.com/openai/v1"
    
)

In [6]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    id_field='doc_id',      # <-- use the existing id so if it's re-run, it will update existing records instead of creating duplicates
    db_path='faq.db'
)

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [10]:
def agent_loop(instructions, question, model="openai/gpt-oss-120b") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [11]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [12]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {
  "query": "join course after start late enrollment policy"
}
iteration #2...
ASSISTANT:
**Short answer:** Yes – you can still join the LLM Zoomcamp even if you discovered it after the official start date.

**What you need to know**

| Aspect | Details |
|--------|---------|
| **Getting started** | The course materials (videos, notebooks, GitHub repo, docs) are always publicly available. You can begin learning right away – there’s no “closed‑door” start date. |
| **Registration** | Registering is only used to gauge interest; it’s not required to access the content or submit homework. If you already signed up you’ll get a welcome email, but even without it you can still work through the lessons. |
| **Certificate eligibility** | To earn the official Zoomcamp certificate you must **submit a completed Capstone project** while the submission window is open. Once the “project submission” form closes, no new certificates will be issued for that cohort.

'**Short answer:**\u202fYes – you can still join the LLM\u202fZoomcamp even if you discovered it after the official start date.\n\n**What you need to know**\n\n| Aspect | Details |\n|--------|---------|\n| **Getting started** | The course materials (videos, notebooks, GitHub repo, docs) are always publicly available. You can begin learning right away – there’s no “closed‑door” start date. |\n| **Registration** | Registering is only used to gauge interest; it’s not required to access the content or submit homework. If you already signed up you’ll get a welcome email, but even without it you can still work through the lessons. |\n| **Certificate eligibility** | To earn the official Zoomcamp certificate you must **submit a completed Capstone project** while the submission window is open. Once the “project submission” form closes, no new certificates will be issued for that cohort. |\n| **Homework & leaderboard** | Homework assignments are optional for the certificate; they only affect you

In [13]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {
  "query": "late enrollment can I still join the course"
}
iteration #2...
ASSISTANT:
**Yes – you can still join the course!**  

- **Joining now:** The course is open‑ended, so you can start working through the videos, notebooks, and assignments right away.  
- **Getting a certificate:** To earn the certificate you must submit a *Capstone project* (and complete the required peer‑reviews) **while the submission form is still open**. As long as the form hasn’t been closed, you can submit your project even if the official “deadline” has passed.  
- **Homework:** The weekly homework assignments are *optional* for certification. Missing or submitting them late won’t stop you from receiving the certificate; they only affect your rank on the leaderboard.  

So, feel free to dive in now, and just make sure you finish and submit your Capstone before the submission window closes.  

---

**Anything else you’d like to explore?**  
- How the Capstone projec

'**Yes – you can still join the course!**  \n\n- **Joining now:** The course is open‑ended, so you can start working through the videos, notebooks, and assignments right away.  \n- **Getting a certificate:** To earn the certificate you must submit a *Capstone project* (and complete the required peer‑reviews) **while the submission form is still open**. As long as the form hasn’t been closed, you can submit your project even if the official “deadline” has passed.  \n- **Homework:** The weekly homework assignments are *optional* for certification. Missing or submitting them late won’t stop you from receiving the certificate; they only affect your rank on the leaderboard.  \n\nSo, feel free to dive in now, and just make sure you finish and submit your Capstone before the submission window closes.  \n\n---\n\n**Anything else you’d like to explore?**  \n- How the Capstone project and peer‑review process work  \n- What resources are available for self‑paced learners  \n- Tips for catching up

## Encouraging multiple searches

In [14]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {
  "query": "Can I join the course enrollment eligibility"
}
iteration #2...
ASSISTANT:
**Yes – you can join the LLM Zoomcamp even if you just discovered it now.**  

- **Open enrollment:** The course does not have a closed‑door registration list. As soon as you sign up (or even without signing up) you can start working through the lectures and homework.  
- **Certificate requirement:** If you’d like to earn the official completion certificate, you’ll need to **submit a capstone project and complete the required peer‑reviews while a live cohort is still accepting submissions**. (You can work through the material at your own pace, but the final project must be turned in before the cohort deadline.)  

So go ahead and start exploring the modules right away. When you feel ready, finish your project and submit it during the next “accepting submissions” window to receive the certificate.

---

Is there anything else you’d like to know—e.g., how to regi

'**Yes – you can join the LLM\u202fZoomcamp even if you just discovered it now.**  \n\n- **Open enrollment:** The course does not have a closed‑door registration list. As soon as you sign up (or even without signing up) you can start working through the lectures and homework.  \n- **Certificate requirement:** If you’d like to earn the official completion certificate, you’ll need to **submit a capstone project and complete the required peer‑reviews while a live cohort is still accepting submissions**. (You can work through the material at your own pace, but the final project must be turned in before the cohort deadline.)  \n\nSo go ahead and start exploring the modules right away. When you feel ready, finish your project and submit it during the next “accepting submissions” window to receive the certificate.\n\n---\n\nIs there anything else you’d like to know—e.g., how to register, cohort dates, project guidelines, or anything about the course content?'

## Restricting off-topic questions

In [15]:
agent_loop(instructions, "what's india capital?")

iteration #1...
function_call: search {
  "query": "India capital"
}
iteration #2...
ASSISTANT:
India’s capital city is **New Delhi**.

Is there anything else you’d like to explore or any other questions you have?


'India’s capital city is **New\u202fDelhi**.\n\nIs there anything else you’d like to explore or any other questions you have?'

In [16]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's india capital?")

iteration #1...
ASSISTANT:
I’m sorry, but I can only help with questions related to the course. If you have any queries about the course material, schedule, assignments, or anything else course‑related, feel free to ask!


'I’m sorry, but I can only help with questions related to the course. If you have any queries about the course material, schedule, assignments, or anything else course‑related, feel free to ask!'